# Capstone - mirrors my deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad-Imran-Toori/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**The paper:** <https://saad-imran-toori.github.io/paper.html>

**What this notebook is, and is not.** It mirrors the paper section by section so the two can be
read against each other. It is deliberately **not** a fourth scan of the warehouse - `w05_model`,
`w06_validation_audit` and `w07_action_playbook` already do that, with outputs committed. Running
30 million rows again would prove nothing that those three do not already prove.

What it does instead is **read the committed metrics receipts and print them**, so every headline
number in the paper is shown to trace to a file in this repo rather than being retyped by hand.
That is the claim worth demonstrating here: not "these numbers are right", but "these numbers came
from somewhere you can check".

**Closing sections carry ML-12** - the five-minute demo outline and the two shareable cuts.


## 1. Question

*The research question and the decision it supports.*

**Where the problem comes from.** FlyRank builds content as infrastructure: it researches, writes
and publishes content directly into a client's website, then watches search data and optimises.
Content that gets found in search ranks, and then quietly decays - rankings slip, clicks drop, and
most teams notice too late. The product already responds to this, flagging pages with a health
score, quick-win tags and needs-attention markers. Those flags are **hand-written rules**:
if-this-then-that, thresholds chosen by a person. They work and they run in production, but rules
run out where signals get many, tangled and shifting, and no trained model has replaced them.

**The research question.** Looking only at what could be known at the end of February, which pages
should an editor review first because they are about to under-capture clicks in March - and can a
learned model order that queue better than the transparent rule already doing the job?

**The decision it supports.** An editor has time for roughly fifty reviews a week against a
portfolio of hundreds of thousands of pages. The output is therefore an **ordered queue**, not a
classification of every page. The cost of a wrong call is not a wrong answer; it is an hour spent
on a page that was fine while a page that needed attention waited.

**What it is not.** Not a claim that editing a page produces clicks. Nothing in this design can
separate an edit from a page recovering on its own, and Section 5 shows that recovery happening.

**One constraint that shaped everything.** The product's own flags are the output of a decision
someone already made. They are used here strictly as **something to beat**, never as model inputs -
a model trained on an existing system's scores learns the old rule, not the world.


In [1]:
# Section 1 is the question and the decision. No computation belongs here.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release.** The FlyRank ML Internship warehouse - a pseudonymised, public-safe release of Google
Search Console and Google Analytics data. Pages and clients appear only as hashed identifiers.

**Tables.** `fact_content_daily_performance`, read one monthly partition at a time, and
`dim_content` for page metadata.

| Window | Months | Role |
|---|---|---|
| Features | January + February 2026 | Everything knowable at the moment of the decision |
| Outcome | March 2026 | The label. Never a feature |
| Sealed | June 2026 | **Never queried**, in any notebook in this project |

**Population.** 203,074 pages appeared in at least one of the three months. The study population is
**40,152 pages across 24 clients**, after four filters:

1. **Present in all three months** - needs January and February for features, March for an outcome.
2. **February impressions >= 500** - a page nobody sees is not an editorial opportunity, and a
   click-through rate computed on twelve impressions is noise.
3. **March impressions >= 100** - March CTR is a rate; on three impressions it carries nothing.
4. **Real February position data** - in this warehouse an average position of zero means *no data*,
   not rank zero. Treating it as a number would silently corrupt every tier assignment.

**Excluded on purpose, not by filtering.** `trend_direction` and `trend_pct` are derived from the
outcome and would leak it. Hashed page and client ids are used for **grouping** during validation
and are never features - an identifier as a feature is a memorisation channel, not a signal.

**Missing stays missing.** The model handles NaN natively. Replacing a missing value with zero
would assert that nothing happened when in fact nothing was recorded.

**Public-safety.** No page URL, client name or search query appears in this notebook, in any
committed output, or in the paper. Full detail: `w03_data_contract.ipynb`.


In [2]:
# Data detail and the eligibility funnel are executed in w03_data_contract.ipynb
# and w05_model.ipynb. Repeating a 30M-row scan here would prove nothing new.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions, stated so they can be argued with.**

- Comparable pages are pages in the same February **search-position tier**. Comparing a top-three
  page's CTR against a deep page's would only rediscover position.
- The editor's binding constraint is attention, so the output must be an ordered queue.
- A page's own recent history is legitimate evidence about its near future.

**Label.** A page **under-captured clicks** if its March CTR fell below the 25th percentile of March
CTR *for its February position tier*. The cut is computed **inside each training fold** and applied
to the held-out fold - never over the whole dataset. A proxy for editorial value, named as one.

**Features.** Eleven numeric and two categorical, all knowable by 28 February: January and February
impressions and clicks, January and February CTR, February average position, a January-to-February
impressions ratio, count of February days visible, word count, content type and main intent - plus
one **injected pure-noise column** as the significance floor for importance analysis.

**Baseline.** The frozen Week-4 rule: expected CTR is the median for the page's February tier;
score is the shortfall multiplied by February impressions. Its tier medians are refitted **inside
every training fold** - computing them once over the whole panel would let the rule see held-out
clients before being tested on them, flattering the baseline.

**Validation.** Five-fold `GroupKFold` on hashed client id; zero client overlap asserted in code.
The model's internal early stopping is **switched off**: it carves a validation set at random with
no knowledge of client, which would leak clients inside every fold.

**The comparison contract.** Same rows, same folds, same metric and K, same tie policy
(score, then February impressions, then page id). Ties matter because the baseline's known failure
is producing identical scores.

**Leakage checks, run rather than assumed.**

1. **Timeline audit** - every feature traced to its window; all sit before March. This clears the
   click features: January and February clicks are knowable at decision time. A temporal boundary,
   not a ban on a data type.
2. **Harness test** - March CTR injected as a feature on purpose. precision@50 went 0.784 to a
   clean **1.000**. A harness that cannot produce that jump cannot detect leakage.
3. **Population audit** - the eligibility filters were checked for outcome-window information. They
   contain some, quantified in Section 5.

Full detail: `w05_model.ipynb` and `w06_validation_audit.ipynb`.


In [3]:
# Methodology is executed in w05_model.ipynb (features, label, baseline, folds)
# and audited in w06_validation_audit.ipynb (timeline, harness test, population).


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Means over five client-grouped folds, 40,152 pages, 24 clients. **Base rate 0.216.**

| Method | precision@20 | precision@50 | sd across folds |
|---|---|---|---|
| Base rate (random ordering) | 0.216 | 0.216 | &mdash; |
| The frozen rule (baseline) | 0.560 | 0.492 | &mdash; |
| Logistic regression | &mdash; | 0.604 | &mdash; |
| Decision tree, depth 3 | &mdash; | 0.692 | &mdash; |
| Random forest | &mdash; | 0.760 | 0.089 |
| **Gradient boosting** | **0.850** | **0.784** | 0.113 |

**It wins in every fold, not on average** - 0.90 / 0.78 / 0.84 / 0.80 / 0.60 against the rule's
0.50 / 0.66 / 0.38 / 0.50 / 0.42.

**Complexity was not free money.** A depth-3 tree already reaches 0.692 and a random forest 0.760
with the *lowest* fold-to-fold spread. Most of the lift comes from using the features at all. Asked
to defend one model in a room, I would defend the forest.

**The honest-split correction.** Re-running under a random split gives 0.836 against the grouped
0.784 - an apparent cost of 0.052 for doing it properly. But the base rate also moves, 0.216 to
0.246, so **0.030 of that 0.052 (57%) is the target shifting, not skill lost**. As a multiple of
base rate the grouped model is slightly *ahead*: 3.63x against 3.40x. I had described the
client-level holdout as a strength for weeks without measuring it, and the first measurement was
wrong in my favour.

**Effect size.** Cohen's *d* for model against rule is **2.04** across five paired folds - large,
but computed on five folds with three holding a single test client, so an order of magnitude rather
than a measurement.

Full detail: `w05_model.ipynb`, `w06_validation_audit.ipynb`.


In [4]:
# Results are produced in w05_model.ipynb and re-measured under a second split
# in w06_validation_audit.ipynb. The receipts are read in section 7 below.


## 5. Limitations

*What this work cannot claim.*

**Scope - the one that changes what the result means.** This ranks pages **still receiving search
impressions during the outcome month**. A page that vanished from search entirely cannot be scored
at all: March CTR is a rate, and a rate needs a denominator. Measured: **979 of 41,234**
February-eligible pages (2.4%) produced no March row, with a median of 864 February impressions -
genuinely visible before they went. They are structurally absent, and they are arguably the pages
most in need of attention.

**The eligibility filter reads the outcome month.** Requiring 100 March impressions removes 102
pages (0.3%), and removing that filter moves the score *up* to 0.828 - so it was never inflating
anything. The requirement that a page appear in all three months is the one that matters, and it is
not patchable: it is a property of choosing a rate as the label.

**Roughly half the signal is click history**, measured once. Removing the four click-derived
features drops the model 0.784 to 0.604 - still ahead of the rule, by 0.112 instead of 0.292.

**Five folds, three holding a single test client.** Spread runs 0.60 to 0.90. Directional.

**One recognisable failure mode.** Every inspected miss in the worst fold had a February CTR of
exactly 0.000% and a March CTR that recovered unaided to 0.090-0.387%. A two-month window cannot
separate "under-converts" from "had one quiet month". No tuning fixes that; a longer history might.

**A negative result, reported because it is a result.** I expected declining pages to be the
priority - the content-decay story. Every momentum bucket sits between 0.198 and 0.242 against a
0.225 base rate: **momentum barely separates pages in either direction.** Below-tier conversion is
what separates them.

**Nothing here is causal.** The model observed which pages under-captured. It says nothing about
whether editing them produces clicks.

**The sealed month is untouched.** June 2026 has never been queried. No claim rests on it.


In [5]:
# Every limitation above is a measured number, not a disclaimer. Sources:
# w06_validation_audit.ipynb (scope, population audit) and
# w07_action_playbook.ipynb (the momentum negative result).


## 6. Ranked recommendations

*The action playbook output - the paper's recommendations section.*

**The queue's most useful property is what it sets aside.** Half the portfolio - **20,061 of 40,152
pages** - falls into `watch_only`, whose under-capture rate is **0.083** against a 0.225 base rate.
Not spending the week in the wrong half is worth more than the ordering of the right half.

**Queue length is the only dial.** There is no probability threshold to tune; K is the threshold.

| K | precision@K | x base rate | Real opportunities | Hours for nothing |
|---|---|---|---|---|
| 10 | 0.900 | 3.99 | 9 | 1 |
| 20 | 0.950 | 4.22 | 19 | 1 |
| 30 | 0.967 | 4.29 | 29 | 1 |
| 50 | 0.940 | 4.17 | 47 | 3 |
| 100 | 0.830 | 3.68 | 83 | 17 |
| 200 | 0.820 | 3.64 | 164 | 36 |

*Pooled across folds - a different measurement from the per-fold 0.784 in Section 4, and not
interchangeable with it.*

| Archetype | Pages | First action |
|---|---|---|
| **Verify first** - real impressions, zero clicks | 6,421 | Confirm reachable, indexed, not a redirect **before** touching copy |
| **Big under-converter** - high impressions, below tier | 4,058 | Rewrite title and meta. Highest value per hour |
| **Growing but diluted** - impressions up >50%, below tier | 4,286 | Check which queries it now matches; tighten to intent |
| **Slipping** - impressions falling, below tier | 1,789 | Refresh and reindex. See the momentum caveat |
| **Thin** - short page, below tier | 717 | Expand to answer fully, then revisit the title |
| **Other under-converter** | 2,820 | Read against query intent; rewrite or refresh |
| **Watch only** | 20,061 | No action |

**Rules for the person using it.** Reason codes exist so the editor can **overrule** the queue. The
queue is **not a safe list** - absence means a low score, no score, or outside the population, and
those are not the same thing. Do not automate the actions. Do not retrain on a drift alert:
data-quality bugs look exactly like drift.

**Largest operational risk is concentration, not accuracy.** The top fifty draws from 8 of 24
clients and a single client holds **44%** of it.

Full detail: `w07_action_playbook.ipynb`.


In [6]:
# The queue, archetypes and cost/value table are produced by
# w07_action_playbook.ipynb, which also writes the two figures the paper embeds.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The paper embeds five figures, all output from executed notebooks rather than redrawn:

| Figure | File | From |
|---|---|---|
| 1 | `ml03-lane.jpg` | The tier rule beating a flat prediction by only 5.6% |
| 2 | `ml04-leakage.png` | R&sup2; 0.037 to 0.874 when a label-derived column is added |
| 3 | `ml09-split.png` | Random vs client-grouped split, raw gap 0.052, adjusted 0.022 |
| 4 | `work/figures/ml10_precision_at_k.png` | precision@K against queue length |
| 5 | `work/figures/ml10_archetype_mix.png` | Archetype mix of the top 50, with accuracy per archetype |

Figures 4 and 5 are regenerated by `w07_action_playbook.ipynb` and committed under `work/figures/`.
Figures 1-3 are screenshots of executed notebook output.

**The receipts.** Rather than retype the paper's numbers here, the cell below reads the committed
metrics file and prints it. If a number in the paper does not appear in a receipt like this one, it
should not be in the paper.


In [7]:
# ---- Read the committed receipts. No warehouse access, no token, no re-run. ----
import json, os

CANDIDATES = [
    "work/outputs/w07_playbook_metrics.json",           # when run from the repo root
    "../outputs/w07_playbook_metrics.json",             # when run from work/notebooks/
    "w07_playbook_metrics.json",
]

path = next((p for p in CANDIDATES if os.path.exists(p)), None)

if path is None:
    print("Receipts file not found locally.")
    print("It is committed at work/outputs/w07_playbook_metrics.json in this repo.")
    print("Clone the repo and re-run this cell from the repo root to read it.")
else:
    with open(path) as f:
        m = json.load(f)

    print("=" * 78)
    print("THE PAPER'S NUMBERS, READ FROM THE COMMITTED RECEIPTS")
    print(f"source: {path}")
    print("=" * 78)

    pop = m["population"]
    print(f"  population              : {pop['pages']:,} pages, {pop['clients']} clients")
    print(f"  feature months          : {', '.join(pop['feature_months'])}")
    print(f"  outcome month           : {pop['label_month']}")
    print(f"  sealed, never queried   : {pop['sealed_month_untouched']}")
    print(f"  scope                   : {pop['scope_note']}")
    print()
    print(f"  base rate               : {m['base_rate']}")
    print(f"  precision@50, per fold  : {m['precision_at_50_per_fold']}")
    print(f"  mean of those           : {m['precision_at_50_per_fold_mean']}"
          "   <- the figure that compares to the baseline")
    print(f"  precision@50, pooled    : {m['precision_at_50_pooled']}"
          "   <- describes the shipped queue")
    print()
    print("  A reader who confuses those two would think the model improved. It did not -")
    print("  they count different sets of 50 pages. The paper says so where both appear.")
    print()
    print("  precision@K:")
    for k, v in m["precision_at_k"].items():
        print(f"    K={k:<4} {v:<6} ({m['lift_over_base_rate'][k]}x base rate)")
    print()
    print("  drift sentinels         :", ", ".join(m["monitoring"]["drift_sentinels"]))
    print("  rejected as sentinel    :", m["monitoring"]["rejected_as_sentinel"])
    print("  retrain cadence         :", m["monitoring"]["retrain_cadence"])
    print()
    print("  queue export            :", m["queue_export"]["path"],
          "- committed:", m["queue_export"]["committed"])
    print("    reason:", m["queue_export"]["reason"])


Receipts file not found locally.
It is committed at work/outputs/w07_playbook_metrics.json in this repo.
Clone the repo and re-run this cell from the repo root to read it.


---

# ML-12 &mdash; Tell the Story

*The three deliverables for ML-12 live below. The first &mdash; tying the findings back to the real
FlyRank content problem &mdash; lives inside the deployed paper itself, in the abstract and the
opening of Section 1, which is where the card says a case study belongs.*

## Five-minute demo outline

*For the Week-8 showcase. Timed for five minutes with one chart.*

**0:00 &ndash; 0:45 &middot; The question, and why it is not the obvious one**

FlyRank publishes content into client sites and flags pages needing attention with hand-written
threshold rules. An editor can review about fifty pages a week out of hundreds of thousands. So
"which pages are underperforming?" is the wrong question &mdash; thousands are. The question is
**which one do I open first**.

**0:45 &ndash; 1:45 &middot; Method, in plain terms**

Features from January and February. Outcome from March: a page under-captured clicks if its March
click-through rate landed in the bottom quarter *of pages at the same search position*. Tier-relative,
or I would only be rediscovering position. Validated with five folds grouped by client, so the model
is never tested on a client it has seen. The rule I had already built was frozen as the baseline, and
both sat the same exam &mdash; same rows, same folds, same metric, same tie policy.

**1:45 &ndash; 2:45 &middot; One chart** &mdash; *precision@K against queue length*

Point at the base-rate line first, then the curve. The message is not "the model is accurate". It is
**a shorter queue is a more accurate queue**: fifty pages yields 47 real finds and 3 wasted hours; a
hundred yields 83 and 17. Doubling the week's work doubles the finds and multiplies the waste by six.

**2:45 &ndash; 3:45 &middot; One honest result**

The model reaches precision@50 of 0.784 against the rule's 0.492, base rate 0.216, and wins in every
fold. Then the caveat, in the same breath: **roughly half that margin is borrowed from click
history** &mdash; remove those four features and it falls to 0.604. And a depth-3 tree I can print on
one page already reaches 0.692. Most of the lift is from using the features at all, not from the
gradient boosting.

**3:45 &ndash; 4:30 &middot; One recommendation**

Hand the editor the top fifty, not the top two hundred. And the most valuable thing the queue does is
not the ordering &mdash; it is setting aside half the portfolio as *watch only*, a group that
under-captures at 0.083 against a 0.225 base rate. Not spending the week in the wrong half beats
optimising the right half.

**4:30 &ndash; 5:00 &middot; The limit, said out loud**

This ranks pages **still visible in search in the outcome month**. Pages that vanished entirely
cannot be scored at all &mdash; a rate needs a denominator &mdash; and those are arguably the ones
most worth flagging. Nothing here says editing a page produces clicks. It orders attention. That is
the whole claim.

*If asked one question, expect: "why grouped folds?" Answer with the measurement, not the principle
&mdash; the raw gap looked like 0.052, but 57% of it was the base rate moving, so client separation
cost far less than it appeared, and I only know that because I measured it.*


## Two shareable cuts

### A. Social post &mdash; about the methodology

> I spent a week proving my own model was less impressive than I'd said.
>
> I'd been describing a client-level holdout as a strength &mdash; testing only on clients the model
> had never seen. Sensible. But I'd never measured what it was worth.
>
> So I built the dishonest version on purpose: same model, same rows, but clients allowed on both
> sides of the split. Precision went from 0.784 to 0.836. The honest split "cost" 0.052.
>
> Except the base rate moved too &mdash; 0.216 to 0.246 &mdash; because the label cut is fitted on
> the training fold and transfers imperfectly to unseen clients. **57% of that gap was the target
> shifting, not skill lost.** Measured as a multiple of its own base rate, the honest split was
> slightly ahead.
>
> The first number was the one I nearly published. Two lessons I'm keeping: report the base rate
> next to every score, and measure a safeguard before you describe it as one.
>
> Full write-up, on three months of real production search data:
> https://saad-imran-toori.github.io/paper.html

### B. Employer-facing summary &mdash; three sentences

> I built a ranking system that tells a content editor which pages to review first, trained on three
> months of real Google Search Console data across 40,152 pages and 24 client sites.
>
> It was validated with client-grouped folds so it is never tested on a client it has seen, compared
> against a frozen transparent baseline on identical terms, and it reached a precision@50 of 0.784
> against the baseline's 0.492 with a base rate of 0.216 &mdash; winning in all five folds.
>
> The write-up states what it cannot claim as prominently as what it can: about half the margin
> rests on click history, pages that left search entirely cannot be scored at all, and nothing in the
> design supports a causal claim about editing.

---

*Paper: <https://saad-imran-toori.github.io/paper.html> &middot;
Repository: <https://github.com/Saad-Imran-Toori/flyrank-ml-internship>*


## Self-check

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** - including the **Abstract** at the top and
      **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut +
      a 3-sentence employer-facing summary.
- [x] The paper's abstract and introduction tie the findings to the real FlyRank content problem -
      hand-written threshold flags that work but run out where signals get tangled - which is the
      case study, living inside the paper rather than in a separate file.
